# Leakage audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yazanjer/An_Explainable_AI_Education/blob/main/notebooks/01_leakage_audit.ipynb)

**Answers:** Editor comments 1 and 2
**Estimated runtime:** 5 min · **Hardware:** CPU
**Quick mode:** set `QUICK_MODE = True` in the setup cell for a fast smoke test.

Demonstrates the failure in the submitted pipeline empirically, so the revision can
report it as a **documented and corrected** finding rather than quietly changing numbers.

Three experiments: a stump on `math_score` alone; a mutual-information ranking of the
old feature matrix; and old-vs-clean performance for the same models.

---


In [ ]:
# --- Environment setup -------------------------------------------------
# Detects Colab, mounts Drive only when in Colab, installs pinned deps.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
QUICK_MODE = True   # set False for the full budget

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = Path("/content/drive/MyDrive/An_Explainable_AI_Education")
    PROJECT.mkdir(parents=True, exist_ok=True)
    if not (PROJECT / "src").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/yazanjer/An_Explainable_AI_Education.git", str(PROJECT)],
                       check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT / "requirements.txt")], check=False)
else:
    PROJECT = Path(os.environ.get("VLPSO_PROJECT_ROOT", Path.cwd().parent))

os.environ["VLPSO_PROJECT_ROOT"] = str(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

from vlpso_xai.config import load_config, set_global_seeds, environment_report
cfg = load_config("quick" if QUICK_MODE else "default")
set_global_seeds(cfg.seed)
cfg.paths.mkdirs()
print("project root:", cfg.paths.root)
print("config:", cfg.config_path.name, "| hash:", cfg.hash()[:12])


In [ ]:
# --- Reproduce the original outcome construction -----------------------
import numpy as np, pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

prim["math_score"] = prim[[f"PV{i}MATH" for i in range(1, 11)]].mean(axis=1)
prim["math_category"] = prim["math_score"].apply(
    lambda s: "Low" if s <= 482 else ("Medium" if s <= 607 else "High"))

for name, (neg, pos) in {"low_vs_medium": ("Low","Medium"),
                         "medium_vs_high": ("Medium","High"),
                         "low_vs_high": ("Low","High")}.items():
    sub = prim[prim.math_category.isin([neg, pos])]
    y = (sub.math_category == pos).astype(int)
    acc = cross_val_score(DecisionTreeClassifier(max_depth=1, random_state=0),
                          sub[["math_score"]], y, cv=5).mean()
    print(f"{name:16s} stump on math_score alone: accuracy = {acc:.4f}")

In [ ]:
# --- The guard raises. This cell is SUPPOSED to fail. ------------------
from vlpso_xai.data.features import assert_no_leakage, LeakageError

old_X = prim.drop(columns=list(prim.select_dtypes(include=["object"]).columns))
try:
    assert_no_leakage(old_X, where="submitted-pipeline")
    print("NO ERROR -- this would be a bug in the guard")
except LeakageError as e:
    print("LeakageError raised as required:\n")
    print(str(e)[:1200])